
#  Kaggle to S3 Upload Notebook

> **Note:** While it's possible to upload files manually through the AWS console or use direct file transfer tools, this notebook provides an automated, scalable method.

- **Why this approach?**
  - Avoids downloading datasets to your local machine.
  - Uses **Google Colab**, which offers **~80 GB of free cloud disk space**, allowing you to download and unzip large datasets directly in the cloud.
  - Then, uploads selected files (like `.csv`) to an **Amazon S3 bucket** using the `boto3` SDK.


##  Step 1: Set up Kaggle API credentials

In [ ]:
# Configure Kaggle authentication without displaying or embedding credentials.
import os
import shutil
from pathlib import Path

source_credentials = Path("kaggle.json")
kaggle_directory = Path.home() / ".kaggle"
target_credentials = kaggle_directory / "kaggle.json"

if source_credentials.is_file():
    kaggle_directory.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_credentials, target_credentials)
    target_credentials.chmod(0o600)
elif not (os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY")):
    raise FileNotFoundError(
        "Configure Kaggle using KAGGLE_USERNAME/KAGGLE_KEY or place kaggle.json in the notebook directory."
    )

## Step 2: Install Required Python Packages

In [ ]:
%pip install boto3 kaggle

##  Step 3: Download Dataset from Kaggle and Upload CSVs to S3

In [ ]:
# Download the Adventure Works dataset and upload its CSV files to S3.
import logging
import os
from pathlib import Path

import boto3
import kaggle

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def main():
    dataset_name = "ukveteran/adventure-works"
    local_path = Path("./data")
    local_path.mkdir(parents=True, exist_ok=True)

    bucket_name = os.getenv("S3_BUCKET_NAME")
    prefix = os.getenv("S3_FOLDER_NAME", "raw-data/").strip("/")
    region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION")
    if not bucket_name:
        raise ValueError("Set S3_BUCKET_NAME before running ingestion")

    try:
        logger.info("Downloading Kaggle dataset %s", dataset_name)
        kaggle.api.authenticate()
        kaggle.api.dataset_download_files(
            dataset=dataset_name,
            path=str(local_path),
            unzip=True
        )

        csv_files = sorted(local_path.rglob("*.csv"))
        if not csv_files:
            raise FileNotFoundError(f"No CSV files found under {local_path}")
        logger.info("Found %d CSV files", len(csv_files))

        # boto3 uses its standard credential provider chain (environment, profile, or role).
        s3 = boto3.client("s3", region_name=region)
        for file_path in csv_files:
            s3_key = f"{prefix}/{file_path.name}" if prefix else file_path.name
            s3.upload_file(str(file_path), bucket_name, s3_key)
            logger.info("Uploaded %s to s3://%s/%s", file_path.name, bucket_name, s3_key)

        logger.info("Ingestion completed successfully")
    except Exception:
        logger.exception("Kaggle-to-S3 ingestion failed")
        raise


main()